# RoBERTa-Large CEFR Evaluator — Joint WCE + Ordinal Loss

This notebook trains and evaluates the **Joint WCE + Ordinal** variant of the CEFR evaluation model used in the thesis.

The evaluator is based on `FacebookAI/roberta-large` and classifies English texts into the six ordered CEFR proficiency levels: A1, A2, B1, B2, C1, and C2.

Training uses the **Evaluation-Judge Dataset**, which is kept disjoint from the Steering Training Dataset used by the generative control methods. A stratified 90/10 training-validation split is used.

The training objective combines:

- **Weighted Cross-Entropy (WCE)** to compensate for CEFR class imbalance; and
- an **ordinal Mean Squared Error penalty** based on the expected class value of the predicted probability distribution.

The complete loss is:

`L_joint = L_WCE + λ L_ordinal`

with `λ = 0.5`.

Evaluation reports Strict Accuracy, Adjacent Accuracy, Macro F1, Mean Absolute Error (MAE), Quadratic Weighted Kappa (QWK), and per-class accuracy.

This model is compared with the WCE-only evaluator variant and is selected as the primary CEFR evaluator used for the generation experiments reported in the thesis.

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn accelerate torch

import os
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, cohen_kappa_score
from torch import nn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00


In [ ]:
# ==========================================
# 1. SETUP & AUTHENTICATION
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# change the hf token
hf_token = "HF_Token"
login(token=hf_token)

MODEL_ID = "FacebookAI/roberta-large"
DATASET_PATH = "/content/drive/MyDrive/Your_Path/evaluation_judge_dataset.csv"

YOUR_HF_REPO_NAME = "MohammadKhosravi/roberta-large-cefr-classifier-JointLoss"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# ==========================================
# 2. DATA LOADING & PREPROCESSING
# ==========================================
print("Loading dataset...")
df = pd.read_csv(DATASET_PATH)

label_map = {"A1": 0, "A2": 1, "B1": 2, "B2": 3, "C1": 4, "C2": 5}
df['label'] = df['cefr'].astype(str).str.strip().str.upper().map(label_map)
df = df.dropna(subset=['label', 'clean_text'])
df['label'] = df['label'].astype(int)

dataset = Dataset.from_pandas(df[['clean_text', 'label']])
dataset = dataset.class_encode_column("label")
dataset = dataset.train_test_split(test_size=0.1, stratify_by_column="label", seed=42)

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_function(examples):
    return tokenizer(examples["clean_text"], padding="max_length", truncation=True, max_length=512)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

Loading dataset...


Stringifying the column:   0%|          | 0/104125 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/104125 [00:00<?, ? examples/s]

Loading Tokenizer...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/93712 [00:00<?, ? examples/s]

Map:   0%|          | 0/10413 [00:00<?, ? examples/s]

In [ ]:
# ==========================================
# 3. JOINT LOSS SETUP (WCE + Ordinal)
# ==========================================
class_counts = df['label'].value_counts().sort_index().values
total_samples = len(df)
num_classes = len(label_map)

class_weights = total_samples / (num_classes * class_counts)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"\nCalculated Class Weights (A1 to C2): {class_weights}")

# 🌟 NEW: Custom Trainer with Joint Loss
class JointLossTrainer(Trainer):
    def __init__(self, lambda_ordinal=0.5, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.lambda_ordinal = lambda_ordinal

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # 1. Weighted Cross-Entropy (Nominal Loss)
        loss_fct_ce = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss_ce = loss_fct_ce(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        # 2. Ordinal Distance Penalty (Expected Value MSE)
        # Convert logits to probabilities
        probs = torch.softmax(logits, dim=-1)
        # Create a tensor of class indices: [0, 1, 2, 3, 4, 5]
        classes = torch.arange(self.model.config.num_labels, device=logits.device, dtype=torch.float)
        # Calculate the expected class value (Continuous prediction)
        expected_values = torch.sum(probs * classes, dim=-1)

        # Calculate Mean Squared Error against the true labels
        loss_fct_ordinal = nn.MSELoss()
        loss_ordinal = loss_fct_ordinal(expected_values, labels.float())

        # 3. Joint Loss Integration
        loss = loss_ce + (self.lambda_ordinal * loss_ordinal)

        return (loss, outputs) if return_outputs else loss


Calculated Class Weights (A1 to C2): [ 0.69416667  0.69416667  0.69416667  0.73756499  3.7611978  17.67226748]


In [ ]:
# ==========================================
# 4. CUSTOM EVALUATION METRICS
# ==========================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    strict_acc = accuracy_score(labels, predictions)
    distances = np.abs(predictions - labels)
    adjacent_acc = np.mean(distances <= 1)
    macro_f1 = f1_score(labels, predictions, average="macro")
    mae = mean_absolute_error(labels, predictions)
    qwk = cohen_kappa_score(labels, predictions, weights="quadratic")

    return {
        "strict_accuracy": strict_acc,
        "adjacent_accuracy": adjacent_acc,
        "macro_f1": macro_f1,
        "mae": mae,
        "qwk": qwk
    }

In [ ]:
# ==========================================
# 5. MODEL INIT & HYPERPARAMETERS (A100 OPTIMIZED)
# ==========================================
print("Loading Model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=6,
    id2label={v: k for k, v in label_map.items()},
    label2id=label_map
).to(device)

# 🌟 UPDATED: A100 80GB Hyperparameters
training_args = TrainingArguments(
    output_dir="./results_joint",
    learning_rate=2e-5,
    per_device_train_batch_size=128,   # 🚀 MASSIVE BATCH SIZE for A100
    per_device_eval_batch_size=128,
    gradient_accumulation_steps=1,    # Processing full batches directly
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    bf16=True,                        # 🚀 BRAIN FLOAT 16: Safer and faster for A100 architectures
    report_to="none"
)

trainer = JointLossTrainer(
    lambda_ordinal=0.5,               # 🌟 The Lambda parameter
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

Loading Model...


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# ==========================================
# 6. EXECUTE TRAINING & SAVE
# ==========================================
print("\n🚀 Starting JOINT LOSS Fine-Tuning on A100...")
trainer.train()

print("\n💾 Saving Best Model...")
trainer.save_model("./best_joint_cefr_model")
tokenizer.save_pretrained("./best_joint_cefr_model")


🚀 Starting JOINT LOSS Fine-Tuning on A100...


Epoch,Training Loss,Validation Loss,Strict Accuracy,Adjacent Accuracy,Macro F1,Mae,Qwk
1,0.459672,0.228528,0.964371,0.990781,0.947226,0.048209,0.973781
2,0.132841,0.190156,0.978488,0.994718,0.966635,0.028522,0.984888
3,0.037006,0.203442,0.980697,0.993470,0.968721,0.027946,0.983866
4,0.016816,0.187967,0.983290,0.994334,0.974739,0.024008,0.986294


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾 Saving Best Model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./best_joint_cefr_model/tokenizer_config.json',
 './best_joint_cefr_model/tokenizer.json')

In [ ]:
print("\n☁️ Pushing to Hugging Face Hub (Corrected)...")

# 1. Inject the repository name directly into the trainer's existing arguments
trainer.args.hub_model_id = YOUR_HF_REPO_NAME

# 2. Re-declare your model card kwargs (without repo_id)
kwargs = {
    "finetuned_from": MODEL_ID,
    "tasks": "text-classification",
    "dataset": "EFCAMDAT (105k rows)",
    "tags": ["cefr", "language-proficiency", "roberta-large", "ordinal-classification", "joint-loss"],
    "model_name": "RoBERTa Large - CEFR Classifier (Joint WCE+Ordinal Method)"
}

# 3. Push to hub (the trainer will now automatically use the hub_model_id we injected)
trainer.push_to_hub(**kwargs)

print(f"✅ Success! Model live at: https://huggingface.co/{YOUR_HF_REPO_NAME}")


☁️ Pushing to Hugging Face Hub (Corrected)...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...s_joint/model.safetensors:   0%|          |  555kB / 1.42GB            

  ...s_joint/training_args.bin:   4%|4         |   222B / 5.26kB            

✅ Success! Model live at: https://huggingface.co/MohammadKhosravi/roberta-large-cefr-classifier-JointLoss


# Per-Class Validation Performance

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("🔮 Extracting predictions on the validation subset...")
# Get predictions and true labels from the evaluation dataset
predictions_output = trainer.predict(tokenized_datasets["test"])
logits = predictions_output.predictions
true_labels = predictions_output.label_ids

# Convert logits to hard class predictions
pred_labels = np.argmax(logits, axis=-1)

# Invert label map to get CEFR names
inv_label_map = {0: "A1", 1: "A2", 2: "B1", 3: "B2", 4: "C1", 5: "C2"}

# Generate a per-class confusion matrix to extract accurate counts
matrix = confusion_matrix(true_labels, pred_labels)

print("\n📊 Custom 'Table 2' Replication - Per-Class Metrics:")
print("-" * 65)
print(f"{'CEFR Level':<12} | {'Validation Count':<18} | {'Accuracy (%)':<14} | {'Error Rate (%)':<10}")
print("-" * 65)

for idx in range(6):
    level_name = inv_label_map[idx]
    total_class_samples = np.sum(matrix[idx])
    correct_predictions = matrix[idx, idx]

    # Calculate exact accuracy and error percentages
    class_accuracy = (correct_predictions / total_class_samples) * 100 if total_class_samples > 0 else 0
    class_error_rate = 100 - class_accuracy

    # 🌟 FIXED: Cleaned up the format specifier to prevent the ValueError
    print(f"{level_name:<12} | {total_class_samples:<18,} | {class_accuracy:<14.2f} | {class_error_rate:<10.2f}")

print("-" * 65)

🔮 Extracting predictions on the validation subset...



📊 Custom 'Table 2' Replication - Per-Class Metrics:
-----------------------------------------------------------------
CEFR Level   | Validation Count   | Accuracy (%)   | Error Rate (%)
-----------------------------------------------------------------
A1           | 2,500              | 99.28          | 0.72      
A2           | 2,500              | 98.48          | 1.52      
B1           | 2,500              | 98.40          | 1.60      
B2           | 2,353              | 97.96          | 2.04      
C1           | 462                | 95.02          | 4.98      
C2           | 98                 | 92.86          | 7.14      
-----------------------------------------------------------------


# Update the Hugging Face Model Card with Per-Class Results

In [ ]:
from huggingface_hub import ModelCard

# 1. Build the Markdown Table String directly from your active 'matrix' variable
markdown_table = "\n\n## Per-Class Evaluation Distribution (Custom Table 2 Replication)\n"
markdown_table += "This table tracks the independent accuracy and error statistics for each individual CEFR level on the validation subset:\n\n"
markdown_table += "| CEFR Level | Validation Count | Accuracy (%) | Error Rate (%) |\n"
markdown_table += "| :--- | :--- | :--- | :--- |\n"

inv_label_map = {0: "A1", 1: "A2", 2: "B1", 3: "B2", 4: "C1", 5: "C2"}

for idx in range(6):
    level_name = inv_label_map[idx]
    total_class_samples = np.sum(matrix[idx])
    correct_predictions = matrix[idx, idx]
    class_accuracy = (correct_predictions / total_class_samples) * 100 if total_class_samples > 0 else 0
    class_error_rate = 100 - class_accuracy

    markdown_table += f"| **{level_name}** | {total_class_samples:,} | {class_accuracy:.2f}% | {class_error_rate:.2f}% |\n"


REPO_TO_UPDATE = "MohammadKhosravi/roberta-large-cefr-classifier-JointLoss"

print("☁️ Pulling current model card from the Hub...")
card = ModelCard.load(REPO_TO_UPDATE)

print("📝 Appending distribution metrics...")
card.text += markdown_table

print("🚀 Uploading updated README.md to your repository...")
card.push_to_hub(REPO_TO_UPDATE)

print("✅ Success! Your custom distribution breakdown is now visible on your model homepage without recalculating predictions!")

☁️ Pulling current model card from the Hub...


README.md:   0%|          | 0.00/2.25k [00:00<?, ?B/s]

📝 Appending distribution metrics...
🚀 Uploading updated README.md to your repository...
✅ Success! Your custom distribution breakdown is now visible on your model homepage without recalculating predictions!


# Independent Evaluation on the Balanced Steering Dataset

This section evaluates the trained Joint-Loss CEFR classifier on a balanced subset derived from the Steering Training Dataset. These texts are disjoint from the Evaluation-Judge Dataset used to train the evaluator, providing an additional independent assessment across equally represented CEFR levels.

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn accelerate torch

import os
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from huggingface_hub import ModelCard
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, cohen_kappa_score, confusion_matrix


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00


In [ ]:
# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# change to a real hf token
hf_token = "HF_Token"
login(token=hf_token)

MODEL_ID = "MohammadKhosravi/roberta-large-cefr-classifier-JointLoss"
# MODEL_ID = "MohammadKhosravi/roberta-large-cefr-classifier-WCE"

# Path to your PPLM dataset subset in Drive
DATASET_PATH = "/content/drive/MyDrive/Your_Path/steering_training_dataset.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running inference for model: {MODEL_ID} on device: {device}")

Running inference for model: MohammadKhosravi/roberta-large-cefr-classifier-JointLoss on device: cuda


In [ ]:
# ==========================================
# 2. DATA LOADING & DETAILED BALANCING
# ==========================================
print("Loading unseen PPLM subset...")
df = pd.read_csv(DATASET_PATH)

label_map = {"A1": 0, "A2": 1, "B1": 2, "B2": 3, "C1": 4, "C2": 5}
df['label'] = df['cefr'].astype(str).str.strip().str.upper().map(label_map)
df = df.dropna(subset=['label', 'clean_text'])
df['label'] = df['label'].astype(int)

# 🌟 CRITICAL BALANCING STEP: Downsample every class to match the minority (C2 = 928)
# Fixed random_state ensures both notebooks evaluate on the EXACT SAME target samples!
SAMPLE_SIZE = 928
print(f"Downsampling dataset to a perfectly balanced profile ({SAMPLE_SIZE} samples per class)...")
balanced_df = df.groupby('label').sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print(f"Total evaluation volume: {len(balanced_df)} rows.")

# Convert to HuggingFace Dataset format
eval_dataset = Dataset.from_pandas(balanced_df[['clean_text', 'label']])

Loading unseen PPLM subset...
Downsampling dataset to a perfectly balanced profile (928 samples per class)...
Total evaluation volume: 5568 rows.


In [ ]:
# ==========================================
# 3. MODEL, TOKENIZER, & BATCH PROCESSING SETUP
# ==========================================
print("Loading Tokenizer and Model from Hub...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device)

def tokenize_function(examples):
    return tokenizer(examples["clean_text"], padding="max_length", truncation=True, max_length=512)

tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# Determine optimal precision format supported by your active GPU
use_bf16 = torch.cuda.is_bf16_supported()

# High-throughput batch inference configuration
eval_args = TrainingArguments(
    output_dir="./inference_temp",
    per_device_eval_batch_size=128,  # 🚀 Massive batching for fast processing
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=eval_args,
    processing_class=tokenizer
)

Loading Tokenizer and Model from Hub...


config.json:   0%|          | 0.00/900 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/388 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Map:   0%|          | 0/5568 [00:00<?, ? examples/s]

In [ ]:
# ==========================================
# 4. EXECUTE INFERENCE & COMPUTE GLOBAL METRICS
# ==========================================
print("\n🚀 Launching batched inference on unseen dataset...")
predictions_output = trainer.predict(tokenized_eval)
logits = predictions_output.predictions
true_labels = predictions_output.label_ids
pred_labels = np.argmax(logits, axis=-1)

# Global Metrics Mathematical Formulations
strict_acc = accuracy_score(true_labels, pred_labels)
adjacent_acc = np.mean(np.abs(pred_labels - true_labels) <= 1)
micro_f1 = f1_score(true_labels, pred_labels, average="micro")
macro_f1 = f1_score(true_labels, pred_labels, average="macro")
mae = mean_absolute_error(true_labels, pred_labels)
qwk = cohen_kappa_score(true_labels, pred_labels, weights="quadratic")
matrix = confusion_matrix(true_labels, pred_labels)


🚀 Launching batched inference on unseen dataset...


In [ ]:

# ==========================================
# 5. GENERATE MARKDOWN REPORTS FOR HF CARD
# ==========================================
inv_label_map = {0: "A1", 1: "A2", 2: "B1", 3: "B2", 4: "C1", 5: "C2"}

# Generate Global Markdown Table
markdown_report = "\n\n## 🧪 Out-of-Distribution Robustness Benchmark (Balanced Unseen Dataset)\n"
markdown_report += "This baseline tracking evaluation was executed on a completely unseen, perfectly balanced data subset downsampled across all categories to match the minority tier baseline profile (928 samples per class; 5,568 rows total):\n\n"

markdown_report += "### Global Metrics\n"
markdown_report += "| Metric | Score |\n| :--- | :---: |\n"
markdown_report += f"| **Strict Accuracy** | {strict_acc*100:.2f}% |\n"
markdown_report += f"| **Adjacent Accuracy** | {adjacent_acc*100:.2f}% |\n"
markdown_report += f"| **Micro F1** | {micro_f1:.4f} |\n"
markdown_report += f"| **Macro F1** | {macro_f1:.4f} |\n"
markdown_report += f"| **Mean Absolute Error (MAE)** | {mae:.4f} |\n"
markdown_report += f"| **Quadratic Weighted Kappa (QWK)** | {qwk:.4f} |\n\n"

# Generate Per-Class Markdown Table
markdown_report += "### Per-Class Diagnostic Breakdown\n"
markdown_report += "| CEFR Level | Evaluated Samples | Accuracy (%) | Error Rate (%) |\n"
markdown_report += "| :--- | :---: | :---: | :---: |\n"

print("\n📊 Out-Of-Distribution Diagnostic Results:")
print("-" * 65)
for idx in range(6):
    level_name = inv_label_map[idx]
    total_samples = np.sum(matrix[idx])
    correct = matrix[idx, idx]
    class_acc = (correct / total_samples) * 100
    class_err = 100 - class_acc

    markdown_report += f"| **{level_name}** | {total_samples:,} | {class_acc:.2f}% | {class_err:.2f}% |\n"
    print(f"Level {level_name:<4} | Evaluated: {total_samples:<5} | Accuracy: {class_acc:.2f}% | Error: {class_err:.2f}%")
print("-" * 65)



📊 Out-Of-Distribution Diagnostic Results:
-----------------------------------------------------------------
Level A1   | Evaluated: 928   | Accuracy: 99.68% | Error: 0.32%
Level A2   | Evaluated: 928   | Accuracy: 98.60% | Error: 1.40%
Level B1   | Evaluated: 928   | Accuracy: 98.38% | Error: 1.62%
Level B2   | Evaluated: 928   | Accuracy: 98.38% | Error: 1.62%
Level C1   | Evaluated: 928   | Accuracy: 95.91% | Error: 4.09%
Level C2   | Evaluated: 928   | Accuracy: 95.47% | Error: 4.53%
-----------------------------------------------------------------


In [ ]:
# ==========================================
# 6. PUSH METRICS TO HUGGING FACE REPOSITORY
# ==========================================
print("\n☁️ Syncing benchmark card to Hugging Face Hub...")
try:
    card = ModelCard.load(MODEL_ID)
    card.text += markdown_report
    card.push_to_hub(MODEL_ID)
    print(f"✅ Success! Robustness metrics appended permanently to: https://huggingface.co/{MODEL_ID}")
except Exception as e:
    print(f"⚠️ Direct repo sync encountered an issue: {e}")
    print("Please save the generated markdown code block manually below for your records:\n")
    print(markdown_report)


☁️ Syncing benchmark card to Hugging Face Hub...


README.md:   0%|          | 0.00/2.75k [00:00<?, ?B/s]

✅ Success! Robustness metrics appended permanently to: https://huggingface.co/MohammadKhosravi/roberta-large-cefr-classifier-JointLoss
